# Databricks CLI, REST API & Python SDK Demo

This notebook demonstrates three ways to interact with your Databricks workspace programmatically: the **CLI**, the **REST API**, and the **Python SDK**. All examples use built-in notebook authentication — no tokens required.

## 1. Databricks CLI Command Reference

```bash
# Check version
databricks --version

# Who am I?
databricks current-user me

# List workspace root
databricks workspace list / --output json

# List clusters
databricks clusters list --output json

# List jobs (limit 3)
databricks jobs list --limit 3 --output json

# Run a job
databricks jobs run-now --job-id 12345

# Get cluster details
databricks clusters get --cluster-id 0911-123456-abc123
```
> **Tip:** Pipe any command through `| jq .` for pretty-printed JSON output.

## 2. Databricks REST API

You can call the Databricks REST API directly using Python's `requests` library. The notebook context provides the workspace URL and a token automatically via `dbutils` and the `WorkspaceClient`.

In [0]:
import requests, json

# Grab workspace host and token from the notebook context
api_host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

headers = {"Authorization": f"Bearer {api_token}"}
print(f"Workspace URL: {api_host}")

In [0]:
# GET /api/2.0/preview/scim/v2/Me  — who am I?
resp = requests.get(f"{api_host}/api/2.0/preview/scim/v2/Me", headers=headers)
user_info = resp.json()
print(json.dumps({"userName": user_info.get("userName"), "displayName": user_info.get("displayName")}, indent=2))

In [0]:
# GET /api/2.1/jobs/list  — list jobs
resp = requests.get(f"{api_host}/api/2.1/jobs/list", headers=headers, params={"limit": 3})
jobs = resp.json().get("jobs", [])

print(f"Found {len(jobs)} job(s):\n")
for j in jobs:
    print(f"  • [{j['job_id']}] {j['settings']['name']}")

In [0]:
# GET /api/2.0/sql/warehouses  — list SQL warehouses
resp = requests.get(f"{api_host}/api/2.0/sql/warehouses", headers=headers)
warehouses = resp.json().get("warehouses", [])

print(f"Found {len(warehouses)} SQL warehouse(s):\n")
for w in warehouses[:5]:
    print(f"  • {w['name']:40s}  state={w['state']}  size={w.get('cluster_size', 'N/A')}")

## 3. Databricks Python SDK

The `databricks-sdk` package provides a typed, Pythonic interface. `WorkspaceClient()` with no arguments picks up notebook credentials automatically.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print(f"Authenticated as: {w.current_user.me().user_name}")

In [0]:
# List jobs using the SDK
for i, job in enumerate(w.jobs.list(limit=5)):
    if i >= 5:
        break
    print(f"  • [{job.job_id}] {job.settings.name}")

In [0]:
# List SQL warehouses using the SDK
for wh in w.warehouses.list():
    print(f"  • {wh.name:40s}  state={wh.state.value}")

In [0]:
# Browse workspace directory
for item in w.workspace.list(f"/Users/{w.current_user.me().user_name}"):
    print(f"  {'📁' if item.object_type.value == 'DIRECTORY' else '📄'} {item.path}")

In [0]:
for i in w.genie.list_spaces().spaces:
    url = f"{w.config.host}/genie/rooms/{i.space_id}"
    print(i.title, i.space_id)
    print(url)

## 4. Genie Spaces: Create & Query Programmatically

The cells below demonstrate creating a **Genie space** via the REST API (the SDK doesn't expose `create_space` yet) and then asking it a question with the SDK's `start_conversation_and_wait`.

In [0]:
import requests, json, uuid
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Auth context ─────────────────────────────────────────────────
api_host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {api_token}", "Content-Type": "application/json"}

# ── Pick the Serverless Starter Warehouse ───────────────────────
wh_id = "91ec2b975e4ebe0b"
print(f"Using warehouse: Serverless Starter Warehouse ({wh_id})")

# ── Build the serialized_space config ───────────────────────────
serialized_space = {
    "version": 2,
    "config": {
        "sample_questions": [
            {
                "id": uuid.uuid4().hex,
                "question": ["What are the top 10 neighborhoods by total incident count?"]
            },
            {
                "id": uuid.uuid4().hex,
                "question": ["What is the average total response time by priority level?"]
            },
            {
                "id": uuid.uuid4().hex,
                "question": ["Which call categories have the longest average on-scene time?"]
            },
            {
                "id": uuid.uuid4().hex,
                "question": ["How many incidents occurred in each precinct?"]
            }
        ]
    },
    "data_sources": {
        "tables": [
            {
                "identifier": "il_sandbox.detroit_911.incidents_bronze",
                "description": [
                    "Detroit 911 emergency incidents data including call details, ",
                    "response times, locations, neighborhoods, and priority levels."
                ],
                "column_configs": [
                    {
                        "column_name": "call_description",
                        "description": ["Description of the 911 call / incident type"],
                        "synonyms": ["incident type", "type of call", "what happened"]
                    },
                    {
                        "column_name": "neighborhood_name",
                        "description": ["Name of the Detroit neighborhood where the incident occurred"],
                        "synonyms": ["neighborhood", "area", "hood"]
                    },
                    {
                        "column_name": "priority",
                        "description": ["Priority/urgency level assigned to the call"],
                        "synonyms": ["urgency", "priority level", "code level"]
                    },
                    {
                        "column_name": "scout_car_area",
                        "description": ["The patrol zone or beat the incident falls within"],
                        "synonyms": ["patrol zone", "beat", "patrol area"]
                    },
                    {
                        "column_name": "total_response_time",
                        "description": ["Total time from call to arrival on scene"],
                        "synonyms": ["response time", "code time", "how fast"]
                    }
                ]
            }
        ]
    },
    "instructions": {
        "text_instructions": [
            {
                "id": uuid.uuid4().hex,
                "content": [
                    "You are a seasoned Detroit 911 Department desk sergeant. ",
                    "Respond to all data inquiries as if you're briefing fellow officers at roll call. ",
                    "Use emergency services terminology and jargon where appropriate — refer to incidents as 'runs', ",
                    "locations as 'scenes', and response times as 'code times'. ",
                    "Keep it professional but with that unmistakable cop cadence. ",
                    "Always provide context about what the numbers mean for public safety."
                ]
            }
        ]
    }
}


In [0]:

# ── Check if a space with this name already exists ──────────────
space_title = f"Detroit 911 Incidents - {w.current_user.me().user_name} - sdk-generated"
existing = [s for s in w.genie.list_spaces().spaces or [] if s.title == space_title]

if existing:
    space_id = existing[0].space_id
    space_url = f"{api_host}/genie/rooms/{space_id}"
    print(f"Space already exists — skipping creation.")
    print(f"   Space ID:    {space_id}")
    print(f"   Title:       {space_title}")
    print(f"   URL:         {space_url}")
else:
    # ── Create the Genie space via REST API ──────────────────────
    payload = {
        "title": space_title,
        "warehouse_id": wh_id,
        "serialized_space": json.dumps(serialized_space)
    }

    resp = requests.post(f"{api_host}/api/2.0/genie/spaces", headers=headers, json=payload)
    if resp.status_code != 200:
        print(f"Error {resp.status_code}: {resp.text}")
        resp.raise_for_status()

    result = resp.json()
    space_id = result["space_id"]
    space_url = f"{api_host}/genie/rooms/{space_id}"

    print(f"\nGenie space created successfully!")
    print(f"   Space ID:    {space_id}")
    print(f"   Title:       {result['title']}")
    print(f"   URL:         {space_url}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Ask a question ──────────────────────────────────────────────
question = "What data is available in this space? What tables and columns can I ask about?"

print(f"Asking Genie: \"{question}\"")

# Start conversation — returns immediately with IDs before polling
wait_op = w.genie.start_conversation(space_id=space_id, content=question)
conv_id = wait_op.conversation_id
conv_url = f"{w.config.host}/genie/rooms/{space_id}/chats/{conv_id}"
print(f"\nConversation link (follow along live): {conv_url}")
print("Polling for response...")

response = wait_op.result()

# ── Extract text & SQL from attachments ─────────────────────────
answer_parts = []
sql_text = ""

if response.attachments:
    for att in response.attachments:
        if hasattr(att, "text") and att.text and hasattr(att.text, "content"):
            answer_parts.append(att.text.content)
        if hasattr(att, "query") and att.query:
            sql_text = att.query.query if hasattr(att.query, "query") else ""

answer_text = "\n".join(answer_parts) if answer_parts else (
    "Response received — no text content returned. Check the Genie space UI for full results."
)
status_val = response.status.value if response.status else "N/A"

print(f"\nStatus: {status_val}")
print(f"Conversation ID: {conv_id}")
print(f"\nAnswer preview:\n{answer_text[:500]}")
if sql_text:
    print(f"\nGenerated SQL:\n{sql_text}")

In [0]:
# ── Build optional SQL block ────────────────────────────────────
sql_block = ""
if sql_text:
    sql_block = f"""
    <div style="margin-top:14px; padding:14px; background:#101729; border-radius:8px;
                border-left:3px solid #4a90d9;">
        <div style="color:#7b8ab8; font-size:11px; text-transform:uppercase;
                    letter-spacing:1px; margin-bottom:6px;">Generated SQL</div>
        <pre style="color:#c8d6e5; margin:0; font-size:13px;
                    white-space:pre-wrap;">{sql_text}</pre>
    </div>"""

# ── Render a Detroit-themed HTML card ────────────────────────────
html = f"""
<div style="font-family:'Segoe UI',system-ui,-apple-system,sans-serif;
            max-width:820px; margin:20px auto;">

    <!-- Header bar -->
    <div style="background:linear-gradient(135deg,#0a1628 0%,#1a2744 100%);
                border-radius:12px 12px 0 0; padding:20px 24px;
                display:flex; align-items:center; gap:14px;">
        <div style="background:#c9a227; color:#0a1628; font-weight:800;
                    font-size:14px; padding:6px 16px; border-radius:4px;
                    letter-spacing:1.5px;">DPD</div>
        <div>
            <div style="color:#e8e8e8; font-size:18px; font-weight:600;">
                Detroit 911 &mdash; Genie Intelligence Briefing</div>
            <div style="color:#7b8ab8; font-size:12px; margin-top:2px;">
                {space_title}</div>
        </div>
    </div>

    <!-- Question -->
    <div style="background:#111827; padding:16px 24px;
                border-left:4px solid #c9a227;">
        <div style="color:#c9a227; font-size:11px; text-transform:uppercase;
                    letter-spacing:1.5px; font-weight:600;">Inquiry</div>
        <div style="color:#e0e0e0; font-size:15px; margin-top:6px;
                    font-style:italic;">\"{question}\"</div>
    </div>

    <!-- Answer body -->
    <div style="background:#0f1729; padding:20px 24px;
                border-radius:0 0 12px 12px;
                border:1px solid #1e2d4a; border-top:none;">
        <div style="color:#4a90d9; font-size:11px; text-transform:uppercase;
                    letter-spacing:1.5px; font-weight:600;
                    margin-bottom:10px;">Briefing Response</div>
        <div style="color:#d1d5db; font-size:14px; line-height:1.75;
                    white-space:pre-wrap;">{answer_text}</div>
        {sql_block}
        <div style="margin-top:16px; padding-top:12px;
                    border-top:1px solid #1e2d4a;
                    color:#4b5563; font-size:11px;">
            Space ID: {space_id} &nbsp;|&nbsp;
            Status: {status_val} &nbsp;|&nbsp;
            <a href="{conv_url}" target="_blank"
               style="color:#4a90d9; text-decoration:none;">Open conversation &rarr;</a>
        </div>
    </div>
</div>
"""
displayHTML(html)

## Comparison Summary

| Aspect | CLI | REST API | Python SDK |
| --- | --- | --- | --- |
| **Best for** | Quick ad-hoc tasks, shell scripts | Language-agnostic integrations | Python apps & notebooks |
| **Auth in notebooks** | Automatic | Manual (token from dbutils) | Automatic (`WorkspaceClient()`) |
| **Typed responses** | JSON text | Raw JSON dicts | Python dataclasses |
| **Error handling** | Exit codes | HTTP status codes | Python exceptions |
| **Pagination** | Built-in (`--limit`) | Manual (next\_page\_token) | Automatic (generators) |